## Inspecting predictions from the demo

Inference has already been run on the test set of about 3,000 doctor–patient conversations, using the same system prompt as in training. The test set was not used in finetuning. Here we simply load and display those results: the first cell downloads the predictions from a public LUMI-O bucket, and the second cell shows one example. By default it shows a random example: **re-run the second cell to see a new one.** To show a specific example, set `POSITION` (its position in the file, counting from 0) or `IDX` (its `idx` in the source dataset) at the top of the cell.

Each record in the `predictions.json` file contains:

| Field | Description |
|---|---|
| `conversation` | The doctor–patient dialogue |
| `reference` | Ground-truth reference answer |
| `base_4b_response` | MedGemma-4B original model response|
| `finetuned_response` | MedGemma-4B fine-tuned model response|
| `base_27b_response` | MedGemma-27B original model response|

In [ ]:
import json
import urllib.request

# Public LUMI-O bucket with the required data - "predictions.json"
BUCKET_URL = "https://462001302.lumidata.eu/health-llm-notebooks-data"

with urllib.request.urlopen(f"{BUCKET_URL}/predictions.json") as f:
    records = json.load(f)

print(f"Loaded {len(records)} predictions")


In [ ]:
# Choose which example to show:
#   POSITION = None      -> a random example (run the cell again for a new one)
#   POSITION = 9         -> the 10th example in the file (counting starts from 0)
POSITION = None


from IPython.display import Markdown, display
import random


def format_conversation(text: str) -> str:
    """Add a blank line after each Doctor/Patient turn for readable Markdown."""
    return text.replace("Doctor:", "\n\n**Doctor:**").replace("Patient:", "\n\n**Patient:**")


def show_example(records: list, position=None) -> None:
    if position is None:
        i = random.randrange(len(records))
        title = "Random example"
    elif 0 <= position < len(records):
        i = position
        title = "Example"
    else:
        raise ValueError(f"POSITION must be between 0 and {len(records) - 1}")

    record = records[i]
    md = f"""
---
## {title}: position {i} of {len(records) - 1}

### 💬 Conversation
{format_conversation(record["conversation"])}

### ✅ Reference (ground truth)
{record["reference"]}

### 🔵 Model 1: Medgemma-4B Base
{record["base_4b_response"]}

### 🟠 Model 2: Medgemma-4B Finetuned
{record["finetuned_response"]}

### 🟢 Model 3: Medgemma-27B Base
{record["base_27b_response"]}
"""
    display(Markdown(md))


show_example(records, position=POSITION)

In [ ]:
# You can download the files in the bucket locally
FILES = ["predictions.json", "validation.jsonl", "README.md"]

import os
import urllib.request

# The files are saved in a bucket_data folder next to this notebook
DOWNLOAD_DIR = os.path.join(os.getcwd(), "bucket_data")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

for name in FILES:
    path = os.path.join(DOWNLOAD_DIR, name)
    if os.path.exists(path):
        print(f"Already downloaded: {path}")
        continue
    urllib.request.urlretrieve(f"{BUCKET_URL}/{name}", path)
    print(f"Downloaded: {path} ({os.path.getsize(path) / 1e6:.1f} MB)")